# 본체 도형+과일패치 재학습 — **cube_v5** (5클래스, round-2 반영)

4종 도형 + `fruit_photo_cube` YOLOv8n. **라운드2 교정 라벨 반영본.**

- 업로드: **`dataset_body_cube_v5.zip`** (= 기존 dataset_body_5class 686장 + **round-2 body 교정본 110장** = 796장 병합. 내부 루트 `dataset/`). ← 라벨링 끝나면 젯슨에서 만들어 줌.
- `yolov8n.pt`서 **새로** 학습(이어학습❌). 결과 best.pt → 젯슨 `models/cube.pt` 교체(본체용).
- ⚠️ **run명 `cube_v5`** — 이전(cube_v4)과 겹치지 않게. 배포 후 held-out로 도형 mAP 회귀 확인.
- 런타임 → GPU 켜고 실행.

In [ ]:
# [셀1] 업로드 + 압축해제 — dataset_body_cube_v5.zip (내부 루트 dataset/)
from google.colab import files
import os, shutil
up = files.upload()
ZIP = next(iter(up))
shutil.rmtree('/content/dataset', ignore_errors=True)
!unzip -o -q "$ZIP" -d /content
print('uploaded:', ZIP, '-> extracted:', sorted(os.listdir('/content/dataset')))

In [ ]:
# [셀2] train/val 분리 + 코랩용 data.yaml 생성 (nc=5, fruit_photo_cube 포함)
import os, glob, random, shutil, yaml
random.seed(0)
ROOT = '/content/dataset'
HELD_OUT_VAL = True      # ← False면 train==val (전체 학습, 검증은 live로)
VAL_FRAC = 0.15
lbl = lambda p: f"/content/dataset/labels/" + os.path.splitext(os.path.basename(p))[0] + ".txt"
pairs = [(i, lbl(i)) for i in sorted(glob.glob(f'/content/dataset/images/*')) if os.path.exists(lbl(i))]
print('pairs:', len(pairs), '(빈 라벨=배경음성 포함)')
if HELD_OUT_VAL:
    random.shuffle(pairs); n = int(len(pairs) * VAL_FRAC)
    for split, items in [('train', pairs[n:]), ('val', pairs[:n])]:
        for s in ('images', 'labels'):
            os.makedirs(f'/content/dataset/{split}/{s}', exist_ok=True)
        for img, lb in items:
            shutil.copy(img, f'/content/dataset/{split}/images/'); shutil.copy(lb, f'/content/dataset/{split}/labels/')
    print('train', len(pairs) - n, '/ val', n)
    data = dict(path=ROOT, train='train/images', val='val/images')
else:
    data = dict(path=ROOT, train='images', val='images')
data.update(nc=5, names=['cube', 'octahedron', 'dodecahedron', 'icosahedron', 'fruit_photo_cube'])
yaml.safe_dump(data, open(f'/content/dataset/data_colab.yaml', 'w'))
print(open(f'/content/dataset/data_colab.yaml').read())

In [ ]:
# [셀3] 학습 — yolov8n서 새로, imgsz=640. run명 cube_v5 (버전 구분)
!pip -q install ultralytics
from ultralytics import YOLO
YOLO('yolov8n.pt').train(data='/content/dataset/data_colab.yaml',
    epochs=100, imgsz=640, batch=16, patience=30, name='cube_v5')

In [ ]:
# [셀4] best.pt 내려받기 → 젯슨 models/cube.pt 교체(본체용)
from google.colab import files
files.download('runs/detect/cube_v5/weights/best.pt')